In [1]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
granularity = 'm'

# --- Date Range (inclusive) ---
START_DATE = '2025-01-01'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---
run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

# --- Rollup Groups (weighted-average aggregation of individual LOB results) ---
ROLLUP_GROUPS = {
    'Franchise Independent': ['AN', 'FLD', 'FRN', 'STG'],
    'nonKMX': ['AN', 'FRN', 'STG', 'FLD', 'ENT'],
    'POS': ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX'],
}

# --- Baselines (per individual LOB) ---
BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235},
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

# --- Model Parameters ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- Impact Score Bounds (floor, ceiling) ---
IMPACT_BOUNDS = {
    'gross_loss_impact': (-20, 10),
    'recovery_impact': (-10, 20),
    'ltv_impact': (-15, 15),
    'apr_impact': (-10, 10),
}

# --- DLA (Dealer Level Adjustment) Current Quarter ---
DLA_CURRENT_QUARTER = '2026 Q1'

# --- Excluded Vintages (per LOB) ---
EXCLUDED_VINTAGES = {}

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: m
Date column: book_date
Period range: 2025-01 to 2026-08
SQL min_date: '2025-01-01'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    """Per-metric population-aware weighted average: each metric's denominator
    only includes amt_financed from accounts where that metric is not NaN."""
    if isinstance(metrics, str):
        valid = group[metrics].notna()
        if valid.any():
            weighted_avg = (group.loc[valid, metrics] * group.loc[valid, 'amt_financed']).sum() / group.loc[valid, 'amt_financed'].sum()
        else:
            weighted_avg = np.nan
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        valid = group[metric].notna()
        if valid.any():
            result_dict[metric] = (
                (group.loc[valid, metric] * group.loc[valid, 'amt_financed']).sum()
                / group.loc[valid, 'amt_financed'].sum()
            )
        else:
            result_dict[metric] = np.nan
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings (e.g. '2025 Q1', '2025 M01')."""
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [ ]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================
# Each table has its own schema-tagged pickle under cache/. When
# run_every_query=False, each cached_sql call reuses the pickle if present
# and falls through to SQL if missing. Delete an individual pickle to force
# a selective refresh.

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_v1.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query.txt', '../../cache/ula_v1.pkl',
            sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
        )
        print('ULA ready')

        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_v1.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")

ULA ready
DLA ready
New recovery ready
ULA records: 3,436,105


In [6]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING + FLAG CREATION
# =============================================================================

# Filter to target LOBs only
ula_df_total = ula_df_total[ula_df_total.lob.isin(LOBS)]

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns using pd.to_period
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")

# =============================================================================
# FLAG CREATION AND DATA REFINEMENT
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_cols = ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']
ula_df_total = ula_df_total.drop(columns=[c for c in dla_cols if c in ula_df_total.columns], errors='ignore')

dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag (derived from ULA job_company) ---
if 'job_company' in ula_df_total.columns:
    warnings.filterwarnings("ignore", category=UserWarning)
    ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
    ula_df_total['driver_flag'] = np.where(
        ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
        1, 0)
    warnings.filterwarnings("default", category=UserWarning)

# --- ULA NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
if 'job_company' in ula_df_total.columns:
    combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
    ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
    ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()
else:
    ula_df_total = ula_df_total.drop_duplicates()

# Convert period to string for vintage-based lookups
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- MTN 4.1 model score transformation (applied AFTER flag creation) ---
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"MTN 4.1 accounts translated: {is_mtn41.sum():,}")

Periods in data: 20
Period range: 2025-01 to 2026-08
ULA after refinement: 244,100
MTN 4.1 accounts translated: 7,338


In [7]:
# =============================================================================
# CELL 7: ACCOUNT-LEVEL RAGU COMPUTATION (vectorized, no vintage loop)
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']

# --- Apply ULA multipliers (all accounts at once, split by KMX vs nonKMX) ---
kmx_mask = ula_df_total.lob == 'KMX'
ula_kmx = get_ula_multiplier_kmx(ula_df_total[kmx_mask].copy())
ula_nonkmx = get_ula_multiplier_nonkmx(ula_df_total[~kmx_mask].copy())
ula_all = pd.concat([ula_kmx, ula_nonkmx])

# --- Merge recovery multiplier (left join to preserve full population) ---
nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')
acct_df = ula_all.merge(
    nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
    on='account_number', how='left'
).drop_duplicates(subset='account_number', keep='first')

# Keep ALL accounts (no bbvalue filter) so model_score covers full portfolio.
# Accounts without bbvalue will have NaN for ltv/ltv_impact/ragu_score.
has_bb = acct_df['bbvalue'].notna() & (acct_df['bbvalue'] > 0)

print(f"Total accounts: {len(acct_df):,}")
print(f"  with bbvalue > 0: {has_bb.sum():,}")
print(f"  with recovery: {acct_df['recovery_multiplier'].notna().sum():,}")

# --- Per-account RAGU computation ---
acct_df['ltv'] = np.where(has_bb, acct_df.amt_financed / acct_df.bbvalue, np.nan)
acct_df['model_score'] = acct_df.cd_model_score

acct_df['unit_loss_score'] = (
    acct_df.model_score
    + (1 - acct_df.loss_multiplier) * mean_unit_loss / unit_loss_to_model_score
)

# Map baselines per LOB
baseline_recovery_map = {lob: cfg['new_recovery_unadjusted'] for lob, cfg in BASELINES.items()}
baseline_ltv_map = {lob: cfg['ltv'] for lob, cfg in BASELINES.items()}
baseline_apr_map = {lob: cfg['apr'] for lob, cfg in BASELINES.items()}

acct_df['baseline_recovery'] = acct_df.lob.map(baseline_recovery_map)
acct_df['baseline_ltv'] = acct_df.lob.map(baseline_ltv_map)
acct_df['baseline_apr'] = acct_df.lob.map(baseline_apr_map)
acct_df['ltv_mult'] = np.where(acct_df.lob == 'KMX', 17 / 0.65, 17)
acct_df['apr_mult'] = np.where(acct_df.lob == 'KMX', 0.7 / 0.65, 0.7)

acct_df['baselined_recovery'] = acct_df.recovery_multiplier / acct_df.baseline_recovery

# RAGU decomposition components (unclipped)
acct_df['gross_loss_impact'] = acct_df.unit_loss_score - acct_df.model_score
acct_df['recovery_impact'] = (
    acct_df.unit_loss_score * mean_unit_loss
    * acct_df.recovery_multiplier * (acct_df.baselined_recovery - 1)
)
acct_df['ltv_impact'] = (acct_df.baseline_ltv / acct_df.ltv - 1) * acct_df.ltv_mult
acct_df['apr_impact'] = (acct_df.baseline_apr - acct_df.apr) / 0.01 * acct_df.apr_mult

# Preserve unclipped for _agg score, then apply bounds to raw score
acct_df['gross_loss_impact_unclipped'] = acct_df['gross_loss_impact'].copy()
acct_df['apr_impact_unclipped'] = acct_df['apr_impact'].copy()

for col, (floor, ceil) in IMPACT_BOUNDS.items():
    acct_df[col] = acct_df[col].clip(lower=floor, upper=ceil)

acct_df['ragu_score'] = (
    acct_df.model_score
    + acct_df.gross_loss_impact
    + acct_df.recovery_impact
    + acct_df.ltv_impact
    + acct_df.apr_impact
)

# =============================================================================
# JENSEN'S CORRECTION: Four variants for recovery and LTV.
#   raw          = per-loan formula, bounded by IMPACT_BOUNDS
#   adj_prop     = proportional scaling using Taylor-based delta
#   adj_contrib  = contribution-based allocation of Taylor delta
#   adj_exact    = proportional scaling using direct exact delta
# Recovery Taylor is exact (degree-3 polynomial). LTV uses direct delta only.
# =============================================================================

def jensen_group_stats(grp):
    M = MODEL_PARAMS['mean_unit_loss']

    scored = grp[grp['recovery_multiplier'].notna() & grp['ltv'].notna()]
    rec_stats = {}
    if len(scored) > 1:
        w = scored['amt_financed'].values
        R = scored['recovery_multiplier'].values
        ULS = scored['unit_loss_score'].values
        B = scored['baseline_recovery'].iloc[0]
        mu_R = np.average(R, weights=w)
        mu_ULS = np.average(ULS, weights=w)
        var_R = np.average((R - mu_R) ** 2, weights=w)
        cov_ULS_R = np.average((ULS - mu_ULS) * (R - mu_R), weights=w)
        mixed_3rd = np.average((ULS - mu_ULS) * (R - mu_R) ** 2, weights=w)
        delta_RI_taylor = M * (mu_ULS * var_R / B + cov_ULS_R * (2 * mu_R / B - 1) + mixed_3rd / B)
        unclipped_RI = ULS * M * R * (R / B - 1)
        agg_RI = np.average(unclipped_RI, weights=w)
        target_RI = M * mu_ULS * mu_R * (mu_R / B - 1)
        delta_RI_exact = agg_RI - target_RI
        rec_stats = {
            'mu_R': mu_R, 'mu_ULS': mu_ULS, 'var_R': var_R,
            'cov_ULS_R': cov_ULS_R, 'mixed_3rd': mixed_3rd,
            'delta_RI_taylor': delta_RI_taylor, 'delta_RI_exact': delta_RI_exact,
            'agg_RI': agg_RI, 'target_RI': target_RI, 'B_rec': B,
        }

    bb = grp[grp['ltv'].notna()]
    ltv_stats = {}
    if len(bb) > 1:
        w = bb['amt_financed'].values
        L = bb['ltv'].values
        B_ltv = bb['baseline_ltv'].iloc[0]
        lm = bb['ltv_mult'].iloc[0]
        mu_L = np.average(L, weights=w)
        unclipped_LI = (B_ltv / L - 1) * lm
        agg_LI = np.average(unclipped_LI, weights=w)
        target_LI = (B_ltv / mu_L - 1) * lm
        delta_LI = agg_LI - target_LI
        ltv_stats = {
            'mu_L': mu_L, 'delta_LI': delta_LI, 'agg_LI': agg_LI,
            'target_LI': target_LI, 'B_ltv': B_ltv, 'ltv_mult_val': lm,
        }

    return {**rec_stats, **ltv_stats}

group_stats = {}
for (vintage, lob), grp in acct_df.groupby(['vintage', 'lob']):
    if lob in BASELINES:
        group_stats[(vintage, lob)] = jensen_group_stats(grp)

# --- Unclipped per-loan values (base for all adjusted columns) ---
acct_df['_ri_unclipped'] = (
    acct_df['unit_loss_score'] * mean_unit_loss
    * acct_df['recovery_multiplier'] * (acct_df['baselined_recovery'] - 1)
)
acct_df['_li_unclipped'] = (acct_df['baseline_ltv'] / acct_df['ltv'] - 1) * acct_df['ltv_mult']

# --- Initialize adjusted columns from unclipped base ---
acct_df['recovery_impact_adj_prop'] = acct_df['_ri_unclipped'].copy()
acct_df['recovery_impact_adj_contrib'] = acct_df['_ri_unclipped'].copy()
acct_df['recovery_impact_adj_exact'] = acct_df['_ri_unclipped'].copy()
acct_df['ltv_impact_adj_prop'] = acct_df['_li_unclipped'].copy()
acct_df['ltv_impact_adj_contrib'] = acct_df['_li_unclipped'].copy()
acct_df['ltv_impact_adj_exact'] = acct_df['_li_unclipped'].copy()

for (vintage, lob), stats in group_stats.items():
    mask = (acct_df.vintage == vintage) & (acct_df.lob == lob)
    scored_mask = mask & acct_df['_ri_unclipped'].notna()
    bb_mask = mask & acct_df['ltv'].notna()

    # --- Recovery adjustments ---
    if 'agg_RI' in stats and stats['agg_RI'] != 0:
        M = MODEL_PARAMS['mean_unit_loss']
        B = stats['B_rec']

        # adj_prop: proportional using Taylor delta
        scale_prop_ri = (stats['agg_RI'] - stats['delta_RI_taylor']) / stats['agg_RI']
        acct_df.loc[scored_mask, 'recovery_impact_adj_prop'] *= scale_prop_ri

        # adj_exact: proportional using exact delta
        scale_exact_ri = stats['target_RI'] / stats['agg_RI']
        acct_df.loc[scored_mask, 'recovery_impact_adj_exact'] *= scale_exact_ri

        # adj_contrib: subtract per-loan share of Taylor delta
        sub = acct_df.loc[scored_mask]
        contrib = M * (
            sub['unit_loss_score'] * (sub['recovery_multiplier'] - stats['mu_R']) ** 2 / B
            + (sub['unit_loss_score'] - stats['mu_ULS'])
              * (sub['recovery_multiplier'] - stats['mu_R'])
              * (2 * stats['mu_R'] / B - 1)
        )
        w = sub['amt_financed']
        avg_contrib = (contrib * w).sum() / w.sum()
        if avg_contrib != 0:
            share = contrib * stats['delta_RI_taylor'] / avg_contrib
            acct_df.loc[scored_mask, 'recovery_impact_adj_contrib'] -= share.values

    # --- LTV adjustments ---
    if 'agg_LI' in stats and stats['agg_LI'] != 0:
        # adj_prop and adj_exact: same (both use exact delta for LTV)
        scale_li = stats['target_LI'] / stats['agg_LI']
        acct_df.loc[bb_mask, 'ltv_impact_adj_prop'] *= scale_li
        acct_df.loc[bb_mask, 'ltv_impact_adj_exact'] *= scale_li

        # adj_contrib: subtract per-loan share of exact delta
        sub = acct_df.loc[bb_mask]
        B_ltv = stats['B_ltv']
        lm = stats['ltv_mult_val']
        mu_L = stats['mu_L']
        contrib_ltv = (B_ltv / sub['ltv'] - B_ltv / mu_L) * lm
        w = sub['amt_financed']
        avg_contrib_ltv = (contrib_ltv * w).sum() / w.sum()
        if avg_contrib_ltv != 0:
            share_ltv = contrib_ltv * stats['delta_LI'] / avg_contrib_ltv
            acct_df.loc[bb_mask, 'ltv_impact_adj_contrib'] -= share_ltv.values

# --- RAGU score agg (uses adj_exact variants) ---
acct_df['ragu_score_agg'] = (
    acct_df.model_score
    + acct_df.gross_loss_impact_unclipped
    + acct_df.recovery_impact_adj_exact
    + acct_df.ltv_impact_adj_exact
    + acct_df.apr_impact_unclipped
)

print(f"Jensen's correction applied to {len(group_stats)} vintage-LOB groups")
print(f"  Recovery: Taylor (3rd-moment exact) + direct exact delta")
print(f"  LTV: direct exact delta (no Taylor)")

# Select output columns
output_cols = [
    'account_number', 'lob', 'vintage', 'book_date', 'app_date', 'model_score',
    'gross_loss_impact',
    'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
    'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
    'apr_impact', 'ragu_score', 'ragu_score_agg', 'amt_financed',
    'ltv', 'apr', 'recovery_multiplier', 'unit_loss_score',
]
acct_df = acct_df[output_cols].copy()

print(f"\nAccount-level RAGU computed: {len(acct_df):,} accounts")
print(f"  with full RAGU (bbvalue + recovery): {acct_df['ragu_score'].notna().sum():,}")
print(f"LOBs: {sorted(acct_df.lob.unique())}")
print(f"Vintages: {acct_df.vintage.nunique()}")
display(acct_df.head(20))

Total accounts: 241,732
  with bbvalue > 0: 239,948
  with recovery: 238,343
Jensen's correction applied to 120 vintage-LOB groups
  Recovery: Taylor (3rd-moment exact) + direct exact delta
  LTV: direct exact delta (no Taylor)

Account-level RAGU computed: 241,732 accounts
  with full RAGU (bbvalue + recovery): 236,888
LOBs: ['AN', 'ENT', 'FLD', 'FRN', 'KMX', 'STG']
Vintages: 20


,account_number,lob,vintage,book_date,app_date,model_score,gross_loss_impact,recovery_impact,recovery_impact_adj_prop,recovery_impact_adj_contrib,recovery_impact_adj_exact,ltv_impact,ltv_impact_adj_prop,ltv_impact_adj_contrib,ltv_impact_adj_exact,apr_impact,ragu_score,ragu_score_agg,amt_financed,ltv,apr,recovery_multiplier,unit_loss_score
0,9.012490e+10,KMX,2025 M02,2025-02-01,2025-01-29,133.0,0.889473,3.370465,-12.288019,3.174409,-12.288019,-11.876030,2.984707,-0.517308,2.984707,-3.230769,122.153139,121.355392,21188.68,2.912533,0.2800,0.626603,133.889473
1,9.012496e+10,KMX,2025 M04,2025-04-03,2025-04-02,150.0,4.325909,-3.390507,1.777654,-3.359011,1.777654,4.046507,-0.315991,-0.220865,-0.315991,-1.518462,153.463447,154.269110,16041.56,1.376958,0.2641,0.532105,154.325909
2,9.012493e+10,KMX,2025 M03,2025-03-10,2025-03-09,150.0,-7.562562,-3.600876,3.667418,-3.841261,3.667418,-13.874528,3.163414,-0.514933,3.163414,-3.230769,121.731265,146.037501,10329.00,3.386557,0.2800,0.524040,142.437438
3,9.012494e+10,KMX,2025 M03,2025-03-18,2025-03-17,136.0,-0.505351,-4.605910,4.691026,-5.371799,4.691026,-1.208970,0.275647,-0.514933,0.275647,-3.230769,126.448999,137.230553,13378.16,1.667060,0.2800,0.501348,135.494649
4,9.012491e+10,KMX,2025 M02,2025-02-25,2025-02-24,139.0,2.682818,-2.644006,9.639499,-2.734622,9.639499,4.161526,-1.045883,-0.517308,-1.045883,-3.230769,139.969569,147.045665,32338.62,1.371734,0.2800,0.539905,141.682818
5,9.012515e+10,KMX,2026 M01,2026-01-05,2026-01-03,152.0,-9.722549,-5.878610,-2.941526,-7.444876,-2.941526,-11.201487,-1.581798,0.387348,-1.581798,-3.230769,121.966585,134.523358,15504.86,2.781141,0.2800,0.480187,142.277451
6,9.012505e+10,KMX,2025 M08,2025-08-11,2025-08-09,148.0,-9.728452,11.630983,5.026300,9.906800,5.026300,3.085880,-0.544219,-0.437280,-0.544219,1.076923,154.065334,143.830552,23679.56,1.422196,0.2400,0.716234,138.271548
7,9.012506e+10,KMX,2025 M08,2025-08-20,2025-08-18,133.0,4.654193,10.575078,4.569993,9.170324,4.569993,3.801609,-0.670444,-0.437280,-0.670444,0.969231,153.000111,142.522974,27104.90,1.388215,0.2410,0.706191,137.654193
8,9.012515e+10,KMX,2025 M12,2025-12-31,2025-12-23,144.0,6.250000,20.000000,12.138120,15.803377,12.138120,10.016770,1.923728,0.555395,1.923728,7.538462,187.805231,171.850310,26902.50,1.149679,0.1800,0.787180,150.250000
9,9.012514e+10,KMX,2025 M12,2025-12-27,2025-12-26,142.0,3.958086,-5.847799,-3.360211,-7.324516,-3.360211,0.342468,0.065771,0.555395,0.065771,-3.230769,137.221985,139.432877,21422.98,1.569449,0.2800,0.483971,145.958086


In [8]:
# =============================================================================
# CELL 8: VINTAGE-LOB AGGREGATION + POS/nonKMX ROLLUPS
# =============================================================================

agg_metrics = [
    'model_score', 'gross_loss_impact',
    'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
    'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
    'apr_impact', 'ragu_score', 'ragu_score_agg', 'ltv', 'apr',
]


def population_aware_agg(grp):
    """Aggregate metrics using the correct non-NaN population for each metric's
    denominator, matching how bareboned_ragu_new.ipynb segments ms_df,
    full_pop_metrics, and recovery_metrics."""
    result = {}

    full_pop = grp
    bb_pop = grp[grp['ltv'].notna()]
    scored_pop = grp[grp['recovery_impact_adj_exact'].notna() & grp['ltv'].notna()]

    full_metrics = ['model_score', 'gross_loss_impact']
    for m in full_metrics:
        sub = full_pop[full_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    bb_metrics = [
        'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
        'apr_impact', 'ltv', 'apr',
    ]
    for m in bb_metrics:
        sub = bb_pop[bb_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    scored_metrics = [
        'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
        'ragu_score', 'ragu_score_agg',
    ]
    for m in scored_metrics:
        sub = scored_pop[scored_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    result['amt_financed'] = full_pop['amt_financed'].sum()
    return pd.Series(result)


# --- Aggregate account-level to vintage-LOB level ---
vintage_lob_df = acct_df.groupby(['vintage', 'lob']).apply(
    population_aware_agg, include_groups=False
).reset_index()

print(f"Vintage-LOB aggregation: {len(vintage_lob_df)} rows "
      f"({vintage_lob_df.vintage.nunique()} vintages x {vintage_lob_df.lob.nunique()} LOBs)")

# --- POS and nonKMX rollups ---
rollup_metrics = [
    'model_score', 'gross_loss_impact',
    'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
    'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
    'apr_impact', 'ragu_score', 'ragu_score_agg', 'ltv', 'apr',
]
for group_name, group_lobs in ROLLUP_GROUPS.items():
    group = vintage_lob_df[vintage_lob_df.lob.isin(group_lobs)].copy()
    rollup = group.groupby('vintage').apply(
        weighted_average_and_sum, rollup_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = group_name
    vintage_lob_df = pd.concat([vintage_lob_df, rollup], ignore_index=True)

print(f"After rollups: {len(vintage_lob_df)} rows across {vintage_lob_df.lob.nunique()} groups")
print(f"Groups: {sorted(vintage_lob_df.lob.unique())}")

display(vintage_lob_df.sort_values(['lob', 'vintage']).head(30))

# --- Diagnostic: population breakdown per vintage-LOB ---
print("\n--- Population breakdown (first 5 vintage-LOB combos) ---")
for (vintage, lob), grp in list(acct_df.groupby(['vintage', 'lob']))[:5]:
    n_total = len(grp)
    n_bb = grp['ltv'].notna().sum()
    n_ragu = grp['ragu_score_agg'].notna().sum()
    af_total = grp['amt_financed'].sum()
    af_bb = grp.loc[grp['ltv'].notna(), 'amt_financed'].sum()
    af_ragu = grp.loc[grp['ragu_score_agg'].notna(), 'amt_financed'].sum()
    print(f"  {lob} {vintage}: {n_total:,} total | {n_bb:,} bbvalue>0 | {n_ragu:,} scored | "
          f"AF total={af_total:,.0f} | AF bb={af_bb:,.0f} | AF scored={af_ragu:,.0f}")

Vintage-LOB aggregation: 120 rows (20 vintages x 6 LOBs)
After rollups: 180 rows across 9 groups
Groups: ['AN', 'ENT', 'FLD', 'FRN', 'Franchise Independent', 'KMX', 'POS', 'STG', 'nonKMX']


,vintage,lob,model_score,gross_loss_impact,ltv_impact,ltv_impact_adj_prop,ltv_impact_adj_contrib,ltv_impact_adj_exact,apr_impact,ltv,apr,recovery_impact,recovery_impact_adj_prop,recovery_impact_adj_contrib,recovery_impact_adj_exact,ragu_score,ragu_score_agg,amt_financed
0,2025 M01,AN,136.837072,1.549964,5.238430,3.590845,3.590845,3.590845,-0.005696,1.601683,0.250081,2.988081,1.925051,1.918451,1.925051,146.687417,143.985500,14046911.28
6,2025 M02,AN,137.099434,1.592689,5.011264,3.471877,3.471877,3.471877,-0.238886,1.610991,0.253413,1.330816,0.462636,0.451699,0.462636,144.735344,142.328932,15142231.96
12,2025 M03,AN,136.694720,1.013862,5.140053,3.510313,3.510313,3.510313,-0.034187,1.607972,0.250488,1.457845,0.481538,0.496911,0.481538,144.286880,141.680833,27450992.22
18,2025 M04,AN,137.574392,1.947546,5.680260,3.827386,3.827386,3.827386,0.275375,1.583492,0.246066,2.240511,1.215933,1.215727,1.215933,147.709155,144.831703,20936616.67
24,2025 M05,AN,138.310065,2.210417,6.210175,4.389264,4.389264,4.389264,0.457148,1.541895,0.243469,2.392337,1.348597,1.347243,1.348597,149.584174,146.719523,20190248.01
30,2025 M06,AN,138.270753,2.731601,5.618327,3.823932,3.823932,3.823932,0.454870,1.583755,0.243502,3.029256,1.934680,1.935192,1.934680,150.113252,147.224281,17056950.45
36,2025 M07,AN,138.466618,2.822211,5.516947,3.668667,3.668667,3.668667,0.678283,1.595652,0.240310,4.098490,3.237059,3.243590,3.237059,151.583848,148.874137,17647443.84
42,2025 M08,AN,138.347606,2.725422,5.766269,3.947371,3.947371,3.947371,0.558684,1.574422,0.242019,3.382379,2.370055,2.370055,2.370055,150.761351,147.930129,17133582.14
48,2025 M09,AN,138.869035,2.464507,5.999327,4.345103,4.345103,4.345103,0.837363,1.545085,0.238038,3.361621,2.299331,2.294322,2.299331,151.526859,148.810345,16787938.75
54,2025 M10,AN,138.811795,2.720606,6.126580,4.694046,4.694046,4.694046,0.629035,1.520233,0.241014,4.372814,3.458321,3.457331,3.458321,152.614787,150.267761,14318739.24



--- Population breakdown (first 5 vintage-LOB combos) ---
  AN 2025 M01: 705 total | 700 bbvalue>0 | 496 scored | AF total=14,046,911 | AF bb=13,931,039 | AF scored=9,837,655
  ENT 2025 M01: 1,671 total | 1,671 bbvalue>0 | 1,180 scored | AF total=38,512,257 | AF bb=38,512,257 | AF scored=27,099,620
  FLD 2025 M01: 464 total | 462 bbvalue>0 | 352 scored | AF total=10,475,203 | AF bb=10,426,087 | AF scored=7,900,463
  FRN 2025 M01: 2,225 total | 2,203 bbvalue>0 | 1,568 scored | AF total=47,591,317 | AF bb=46,948,091 | AF scored=32,694,568
  KMX 2025 M01: 4,015 total | 4,014 bbvalue>0 | 3,855 scored | AF total=98,513,480 | AF bb=98,449,486 | AF scored=94,400,320


In [9]:
# =============================================================================
# CELL 9: EXCEL EXPORT -- INDIVIDUAL + AGGREGATED
# =============================================================================
import openpyxl
EXCEL_SHEET_MAP = {'q': 'Individual (Q)', 'm': 'Individual (M)', 'w': 'Individual (W)'}
AGG_SHEET_MAP   = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = '../output/ragu_individual.xlsx'

METRIC_ROWS = [
    ('Model Score',                         'model_score'),
    ('Gross Loss Impact',                   'gross_loss_impact'),
    ('Recovery Impact',                     'recovery_impact'),
    ('Recovery Impact (Adj Prop)',          'recovery_impact_adj_prop'),
    ('Recovery Impact (Adj Contrib)',       'recovery_impact_adj_contrib'),
    ('Recovery Impact (Adj Exact)',         'recovery_impact_adj_exact'),
    ('LTV Impact',                          'ltv_impact'),
    ('LTV Impact (Adj Prop)',              'ltv_impact_adj_prop'),
    ('LTV Impact (Adj Contrib)',           'ltv_impact_adj_contrib'),
    ('LTV Impact (Adj Exact)',             'ltv_impact_adj_exact'),
    ('APR Impact',                          'apr_impact'),
    ('RAGU Score',                          'ragu_score'),
    ('RAGU Score (Agg)',                    'ragu_score_agg'),
    ('Amount Financed',                     'amt_financed'),
    ('Weighted LTV',                        'ltv'),
    ('Weighted APR',                        'apr'),
]

# --- Individual account-level sheet ---
indiv_sheet = EXCEL_SHEET_MAP[granularity]

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if indiv_sheet in wb.sheetnames:
        del wb[indiv_sheet]
    ws = wb.create_sheet(indiv_sheet, 0)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = indiv_sheet

header = ['account_number', 'lob', 'vintage', 'book_date', 'app_date',
          'model_score', 'gross_loss_impact',
          'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
          'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
          'apr_impact', 'ragu_score', 'ragu_score_agg', 'amt_financed',
          'ltv', 'apr', 'recovery_multiplier', 'unit_loss_score']
for col_idx, col_name in enumerate(header, start=1):
    ws.cell(row=1, column=col_idx, value=col_name)

for row_idx, row in enumerate(acct_df.itertuples(index=False), start=2):
    for col_idx, val in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=val)

print(f"Individual sheet '{indiv_sheet}': {len(acct_df):,} rows")

# --- Aggregated vintage-LOB sheet (same pivot format as bareboned) ---
agg_sheet = AGG_SHEET_MAP[granularity]
if agg_sheet in wb.sheetnames:
    del wb[agg_sheet]
ws_agg = wb.create_sheet(agg_sheet)

sorted_vintages = sorted(vintage_lob_df['vintage'].unique())
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())
current_row = 1

for lob in all_export_lobs:
    lob_data = vintage_lob_df[vintage_lob_df.lob == lob].set_index('vintage')

    ws_agg.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws_agg.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws_agg.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws_agg.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Aggregated sheet '{agg_sheet}': {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")
print(f"Saved to {EXCEL_OUTPUT}")

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# CELL 10: SANDBOX EXPORT -- RAGU INPUTS (ACCOUNT-LEVEL)
# =============================================================================

TABLE = 'sandbox.gl_ragu_individual'
BATCH_SIZE = 1000

def _fmt(v, dp=4):
    return 'NULL' if pd.isna(v) else f'{v:.{dp}f}'

upload_df = acct_df[[
    'account_number', 'lob', 'book_date', 'model_score', 'gross_loss_impact',
    'recovery_multiplier', 'ltv', 'apr', 'amt_financed', 'ragu_score'
]].copy()
upload_df['gross_loss_ragu'] = upload_df['model_score'] + upload_df['gross_loss_impact']
upload_df = upload_df.reset_index(drop=True)

print(f"Rows to upload: {len(upload_df):,}")
display(upload_df.head(10))

if granularity != 'm':
    print(f"WARNING: Sandbox upload skipped (granularity='{granularity}'). "
          "Only runs for monthly ('m') granularity.")
else:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {TABLE};")
        cur.execute(f"""
            CREATE TABLE {TABLE} (
                account_number      BIGINT,
                lob                 VARCHAR(25),
                book_date           DATE,
                model_score         FLOAT,
                gross_loss_impact   FLOAT,
                gross_loss_ragu     FLOAT,
                ragu_score          FLOAT,
                recovery_multiplier FLOAT,
                ltv                 FLOAT,
                apr                 FLOAT,
                amt_financed        FLOAT
            );
        """)
        conn.commit()

        for start in range(0, len(upload_df), BATCH_SIZE):
            batch = upload_df.iloc[start:start + BATCH_SIZE]
            values = ", ".join(
                f"({int(r.account_number)}, '{r.lob}', '{r.book_date.strftime('%Y-%m-%d')}', "
                f"{_fmt(r.model_score)}, {_fmt(r.gross_loss_impact)}, {_fmt(r.gross_loss_ragu)}, "
                f"{_fmt(r.ragu_score)}, "
                f"{_fmt(r.recovery_multiplier)}, {_fmt(r.ltv)}, {_fmt(r.apr)}, {_fmt(r.amt_financed, 2)})"
                for r in batch.itertuples(index=False)
            )
            cur.execute(f"INSERT INTO {TABLE} VALUES {values}")
            conn.commit()

        print(f"Uploaded {len(upload_df):,} rows to {TABLE}")

        cur.execute("CALL sandbox.util_table_grant('gl_ragu_individual')")
        conn.commit()
        print("Granted access on sandbox.gl_ragu_individual")

Rows to upload: 241,732


,account_number,lob,book_date,model_score,gross_loss_impact,recovery_multiplier,ltv,apr,amt_financed,ragu_score,gross_loss_ragu
0,9.012490e+10,KMX,2025-02-01,133.0,0.889473,0.626603,2.912533,0.2800,21188.68,122.153139,133.889473
1,9.012496e+10,KMX,2025-04-03,150.0,4.325909,0.532105,1.376958,0.2641,16041.56,153.463447,154.325909
2,9.012493e+10,KMX,2025-03-10,150.0,-7.562562,0.524040,3.386557,0.2800,10329.00,121.731265,142.437438
3,9.012494e+10,KMX,2025-03-18,136.0,-0.505351,0.501348,1.667060,0.2800,13378.16,126.448999,135.494649
4,9.012491e+10,KMX,2025-02-25,139.0,2.682818,0.539905,1.371734,0.2800,32338.62,139.969569,141.682818
5,9.012515e+10,KMX,2026-01-05,152.0,-9.722549,0.480187,2.781141,0.2800,15504.86,121.966585,142.277451
6,9.012505e+10,KMX,2025-08-11,148.0,-9.728452,0.716234,1.422196,0.2400,23679.56,154.065334,138.271548
7,9.012506e+10,KMX,2025-08-20,133.0,4.654193,0.706191,1.388215,0.2410,27104.90,153.000111,137.654193
8,9.012515e+10,KMX,2025-12-31,144.0,6.250000,0.787180,1.149679,0.1800,26902.50,187.805231,150.250000
9,9.012514e+10,KMX,2025-12-27,142.0,3.958086,0.483971,1.569449,0.2800,21422.98,137.221985,145.958086


Uploaded 241,732 rows to sandbox.gl_ragu_individual_nofraud
Granted access on sandbox.gl_ragu_individual_nofraud


In [ ]:
# =============================================================================
# CELL 11: ROW COUNT BY LOB
# =============================================================================

lob_counts = upload_df.groupby('lob').size().reset_index(name='row_count')
lob_counts.loc[len(lob_counts)] = ['TOTAL', lob_counts['row_count'].sum()]
display(lob_counts)

,lob,row_count
0,AN,49442
1,ENT,101365
2,FLD,32961
3,FRN,116750
4,KMX,429414
5,STG,88248
6,TOTAL,818180


In [ ]:
# =============================================================================
# CELL 10: VALIDATION -- Compare ragu_score_agg rollup vs bareboned output
# =============================================================================

bareboned_path = '../output/barebones_ragu.xlsx'
if os.path.exists(bareboned_path):
    bb_sheet_map = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
    bb_sheet = bb_sheet_map.get(granularity)

    wb_bb = openpyxl.load_workbook(bareboned_path, read_only=True)
    if bb_sheet and bb_sheet in wb_bb.sheetnames:
        ws_bb = wb_bb[bb_sheet]
        rows = list(ws_bb.iter_rows(values_only=True))
        wb_bb.close()

        bb_records = []
        i = 0
        while i < len(rows):
            lob_name = rows[i][0]
            if lob_name and isinstance(lob_name, str) and lob_name in (LOBS + list(ROLLUP_GROUPS.keys())):
                vintages_row = rows[i][1:]
                ragu_row_idx = None
                for offset in range(1, 12):
                    if i + offset < len(rows) and rows[i + offset][0] == 'RAGU Score':
                        ragu_row_idx = i + offset
                        break
                if ragu_row_idx is not None:
                    ragu_values = rows[ragu_row_idx][1:]
                    for v_name, r_val in zip(vintages_row, ragu_values):
                        if v_name and r_val is not None:
                            bb_records.append({'lob': lob_name, 'vintage': str(v_name), 'bareboned_ragu': float(r_val)})
            i += 1

        bb_df = pd.DataFrame(bb_records)

        indiv_agg = vintage_lob_df[['lob', 'vintage', 'ragu_score_agg',
                                     'recovery_impact_adj_exact', 'ltv_impact_adj_exact']].copy()
        comparison = indiv_agg.merge(bb_df, on=['lob', 'vintage'], how='inner')
        comparison['ragu_diff'] = comparison['ragu_score_agg'] - comparison['bareboned_ragu']

        print("=== Validation: ragu_score_agg vs bareboned_ragu_new.ipynb ===")
        print(f"Matched vintage-LOB combinations: {len(comparison)}")
        print(f"\nRAGU Score Difference (individual_agg - bareboned):")
        print(f"  Mean:     {comparison['ragu_diff'].mean():.6f}")
        print(f"  MAE:      {comparison['ragu_diff'].abs().mean():.6f}")
        print(f"  Max abs:  {comparison['ragu_diff'].abs().max():.6f}")
        print(f"  Std:      {comparison['ragu_diff'].std():.6f}")

        pd.set_option('display.float_format', '{:.4f}'.format)
        display(comparison.sort_values(['lob', 'vintage']))
    else:
        print(f"Sheet '{bb_sheet}' not found in {bareboned_path}")
        wb_bb.close()
else:
    print(f"Bareboned output file '{bareboned_path}' not found. Run bareboned_ragu_new.ipynb first to generate comparison data.")

ValueError: could not convert string to float: '=AA95-W95'